# LiRPA refinement ablation

Comparison of the same Eq. (1) anchor neighborhoods under geometry-only projection, targeted PGD, and CertCF refinement. Quality metrics are shown both on each method's successes and on paired successes.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS = ROOT / 'results' / 'lirpa_refinement_ablation'
queries = pd.read_parquet(RESULTS / 'lirpa_refinement_queries.parquet')
summary = pd.read_parquet(RESULTS / 'lirpa_refinement_summary.parquet')
empirical = pd.read_parquet(RESULTS / 'empirical_robustness.parquet')
assert not queries.duplicated(['case_id', 'query_position', 'method']).any()
queries.shape, summary.shape

## Completion and all-query outcomes

In [ ]:
completion = (queries.groupby(['case_id', 'method'])
              .agg(rows=('query_position', 'size'), success_pct=('success', lambda x: 100*x.mean()),
                   timeout_pct=('timed_out', lambda x: 100*x.mean()),
                   feasibility_pct=('domain_feasible', lambda x: 100*x.mean()))
              .reset_index())
completion

## Per-case benchmark table

In [ ]:
columns = ['case_id', 'method', 'success_rate', 'timeout_rate', 'mean_l1', 'mean_l2',
           'mean_l0', 'mean_mad_l1', 'mean_redundancy', 'mean_log10_lof', 'mean_isolation_forest_score',
           'empirical_target_rate', 'mean_certified_l1_radius',
           'mean_runtime_seconds', 'mean_certcf_to_pgd_l1_ratio']
summary[[c for c in columns if c in summary]].sort_values(['case_id', 'method'])

## Paired quality on shared successes

In [ ]:
quality_metrics = ['l1_distance', 'l2_distance', 'l0_changed', 'mad_l1_distance', 'redundancy', 'log10_lof',
                   'isolation_forest_score', 'certified_l1_radius', 'runtime_seconds']
paired_all = (queries[queries.shared_success_all]
              .groupby(['case_id', 'method'])[quality_metrics].mean().reset_index())
paired_pgd_certcf = (queries[queries.method.isin(['anchor_pgd', 'certcf']) &
                              queries.shared_success_pgd_certcf.fillna(False)]
                     .groupby(['case_id', 'method'])[quality_metrics].mean().reset_index())
display(paired_all)
display(paired_pgd_certcf)

## Synthetic32 depth trends

In [ ]:
synthetic = summary[summary.kind == 'synthetic32'].copy()
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for metric, axis, title in [
    ('success_rate', axes[0,0], 'Target success'),
    ('mean_l1', axes[0,1], 'Mean $L_1$ distance'),
    ('mean_certified_l1_radius', axes[1,0], 'Certified $L_1$ radius'),
    ('mean_runtime_seconds', axes[1,1], 'Online runtime (s)')]:
    sns.lineplot(data=synthetic, x='depth', y=metric, hue='method', marker='o', ax=axis)
    axis.set_title(title)
plt.tight_layout()

## Robustness curves

In [ ]:
curves = empirical.groupby(['case_id', 'method', 'sigma']).target_rate.mean().reset_index()
grid = sns.relplot(data=curves, x='sigma', y='target_rate', hue='method',
                   col='case_id', col_wrap=3, kind='line', marker='o', height=3.2)
grid.set_axis_labels('Gaussian $\sigma$', 'Target preservation rate')

## Failure and resource diagnostics

In [ ]:
failure_breakdown = (queries[~queries.success]
                     .assign(reason=lambda d: np.where(d.timed_out, 'timeout',
                         np.where(~d.candidate_found, 'no candidate',
                         np.where(~d.domain_feasible, 'infeasible', 'wrong target'))))
                     .groupby(['case_id', 'method', 'reason']).size()
                     .rename('count').reset_index())
display(failure_breakdown)
build_rows = []
for path in sorted((RESULTS / 'cases').glob('*/certcf/build.json')):
    build_rows.append(json.loads(path.read_text()))
builds = pd.DataFrame(build_rows)
builds